# SFT for LLMs

Supervised fine-tuning (SFT) trains a language model on examples of the replies we want. We start from SmolLM2-135M-Instruct, inspect its chat tokens, and teach it a fixed answer style. The questions stay in the context. Only assistant completions contribute to the loss.

This is the supervised step described by [Ouyang et al. (2022)](https://arxiv.org/abs/2203.02155). The small model comes from [Allal et al., *SmolLM2: When Smol Goes Big* (2025)](https://arxiv.org/abs/2502.02737). You need Python and some familiarity with next-token prediction; the JAX gradient check below works directly on the logits.

The full notebook expects one NVIDIA GPU. The first run downloads the checkpoint (about 270 MB) and its tokenizer. We write all the training examples here, so there is no dataset download.


## Install dependencies

Install Dew from GitHub with the CUDA and streaming extras. The checkpoint loader reads Hugging Face's safetensors files.


In [ ]:
%pip install -q "dewml[cuda13,streaming] @ git+https://github.com/AshishKumar4/dew"


## Settings

A training window has `SEQUENCE_LENGTH` inputs and one extra token for the shifted targets. The learning rate is small because we start from trained weights. `MAX_NUMBER` controls how many arithmetic examples we write.


In [1]:
import os

CHECKPOINT = "HuggingFaceTB/SmolLM2-135M-Instruct"
SEQUENCE_LENGTH = 128
BATCH_SIZE = 4
STEPS = 80
LEARNING_RATE = 1e-4
MAX_NEW_TOKENS = 40
MAX_NUMBER = 8
HELD_OUT = 3
SYSTEM_PROMPT = "Be brief."
SIGN_OFF = "Dew done."
SEED = 0

# The tutorial test sets DEW_TUTORIAL_SMOKE=1 to run every cell in minutes on a CPU.
if os.environ.get("DEW_TUTORIAL_SMOKE") == "1":
    SEQUENCE_LENGTH, BATCH_SIZE, STEPS, MAX_NEW_TOKENS = 64, 1, 2, 8
    MAX_NUMBER, HELD_OUT = 3, 1


In [2]:
import jax
import jax.numpy as jnp
import numpy as np

from dew.data import HFTokenizer
from dew.interop import PretrainedDecoder

print(jax.devices())


/home/ubuntu/envs/dew-cuda12-629eb2e9fac9/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[CudaDevice(id=0)]


## Chat templates

A decoder sees token IDs, so the roles and turn boundaries need a text representation. Each instruct model has its own chat template. Use the template and vocabulary that came with the checkpoint.

`PretrainedDecoder.load` returns the model, its weights and its text processor together. The model computes in bfloat16 and keeps its parameters in float32, as in [notebook 08](08-load-a-pretrained-decoder.ipynb).


In [3]:
model = PretrainedDecoder.load(CHECKPOINT, dtype=jnp.bfloat16, max_seq_len=256)
tokenizer = HFTokenizer(CHECKPOINT).tokenizer
print(f"{sum(x.size for x in jax.tree.leaves(model.variables)) / 1e6:.0f}M parameters")


Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

Fetching 11 files: 100%|██████████| 11/11 [00:00<00:00, 1695.54it/s]

Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

Fetching 11 files: 100%|██████████| 11/11 [00:00<00:00, 1774.58it/s]

[dry-run] Fetching 25 files:   0%|          | 0/25 [00:00<?, ?it/s]

...

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Fetching 1 files: 100%|██████████| 1/1 [00:00<00:00, 1073.54it/s]

Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

Fetching 11 files: 100%|██████████| 11/11 [00:00<00:00, 1598.11it/s]

135M parameters


Here is one conversation as a list of messages. Render it first as text, then as IDs. The template supplies markers such as `<|im_start|>` and `<|im_end|>`, the role names and the newlines. They are part of the input the model learned to read.


In [4]:
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": "2 + 3?"},
    {"role": "assistant", "content": f"5; {SIGN_OFF}"},
]
rendered_text = tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=False)
template_ids = tokenizer.apply_chat_template(
    messages, tokenize=True, return_dict=False, add_generation_prompt=False)
print(rendered_text)
print("token IDs:", template_ids)


<|im_start|>system
Be brief.<|im_end|>
<|im_start|>user
2 + 3?<|im_end|>
<|im_start|>assistant
5; Dew done.<|im_end|>

token IDs: [1, 9690, 198, 6077, 5453, 30, 2, 198, 1, 4093, 198, 34, 1232, 216, 35, 47, 2, 198, 1, 520, 9531, 198, 37, 43, 45284, 2294, 30, 2, 198]


## How SFT rows are built

`ChatData` accepts a source with a `messages` column, including a tuple of records already in memory. It parses each row as a `Conversation` and uses `render_conversation` to produce `text` (IDs) and `text_roles` (one `Role` per token).

`long="truncate"` cuts an overlong conversation on the right. `"drop"` removes it, and `"chunk"` splits it into separate pieces without overlap. A cut can remove the assistant reply; a chunk loses the context in earlier pieces. Our examples fit in a window.

Eight copies of this short conversation give us enough rows to inspect packing. `read="memory"` prepares them in host memory; `Loading(workers=0)` keeps this small example in the notebook process.


In [5]:
from dew.config.data import ChatData
from dew.data import Loading
from dew.data.chat import Conversation, Role, render_conversation
from dew.data.dataset import DataPartition

demo_source = ({"messages": messages},) * 8
demo_data = ChatData(
    demo_source, tokenizer=CHECKPOINT, seq_len=SEQUENCE_LENGTH, long="truncate",
).load(batch=1, read="memory", loading=Loading(workers=0), seed=SEED)
batch = next(iter(demo_data.train(DataPartition())))

ids, row_roles = render_conversation(
    tokenizer, Conversation.parse(messages), CHECKPOINT)
np.testing.assert_array_equal(ids, template_ids)
print(f"{'position':>8} {'id':>6} {'role':>10}  token")
for position, (token, role) in enumerate(zip(ids, row_roles, strict=True)):
    print(f"{position:8} {token:6} {Role(int(role)).name.lower():>10}  "
          f"{tokenizer.decode([int(token)])!r}")


Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

Fetching 11 files: 100%|██████████| 11/11 [00:00<00:00, 1739.92it/s]

position     id       role  token
       0      1     system  '<|im_start|>'
       1   9690     system  'system'
       2    198     system  '\n'
       3   6077     system  'Be'
       4   5453     system  ' brief'
       5     30     system  '.'
       6      2     system  '<|im_end|>'
       7    198     system  '\n'
       8      1       user  '<|im_start|>'
       9   4093       user  'user'
      10    198       user  '\n'
      11     34       user  '2'
      12   1232       user  ' +'
      13    216       user  ' '
      14     35       user  '3'
      15     47       user  '?'
      16      2       user  '<|im_end|>'
      17    198       user  '\n'
      18      1        pad  '<|im_start|>'
      19    520        pad  'ass'
      20   9531        pad  'istant'
      21    198        pad  '\n'
      22     37  assistant  '5'
      23     43  assistant  ';'
      24  45284  assistant  ' Dew'
      25   2294  assistant  ' done'
      26     30  assistant  '.'
      27      2  

## Loss masks on assistant turns

`LMObjective(..., loss_role=Role.ASSISTANT)` counts only targets labelled `ASSISTANT`. The role belongs to the target, one position after the input token that predicts it.

The renderer leaves an assistant's opening header at `Role.PAD`, so we learn its completion and closing turn marker. Real headers still have a nonzero segment ID. `Role.PAD` by itself does not mean that a token is absent.

The mask below also excludes padding and transitions between conversations. We will look at those segment boundaries next.


In [6]:
from dew.objectives.lm import LMObjective

objective = LMObjective(model, SEQUENCE_LENGTH, loss_role=Role.ASSISTANT)
tokens = np.asarray(batch["text"])
roles = np.asarray(batch["text_roles"])
segments = np.asarray(batch["text_coordinates"].segments)
mask = ((roles[:, 1:] == Role.ASSISTANT)
        & (segments[:, 1:] == segments[:, :-1])
        & (segments[:, 1:] != 0))
real_tokens = int((segments[0] != 0).sum())
print("m_t for the real targets:", mask[0, :real_tokens - 1].astype(int).tolist())
print(f"scored targets: {int(mask[0].sum())} of {real_tokens - 1} real next-token slots")
print("scored padding targets:", int(mask[0, real_tokens - 1:].sum()))


m_t for the real targets: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1]
scored targets: 28 of 115 real next-token slots
scored padding targets: 0


## Packing SFT rows

`ChatData` places rendered conversations first-fit into windows of `seq_len + 1` tokens. Each batch carries `text_coordinates`, the window's layout: its `segments` identify each conversation inside a window, with 0 marking padding, and its `positions` restart at 0 for each segment.

Causal attention can read only earlier tokens in the same segment. A target counts only when its input and target have the same nonzero segment ID. The last token of one conversation therefore never trains the first token of the next one.


In [7]:
print("packed token IDs:", tokens[0].tolist())
print("segment IDs:     ", segments[0].tolist())
print("positions:       ", np.asarray(batch["text_coordinates"].positions)[0].tolist())
for segment in np.unique(segments[0]):
    if segment != 0:
        columns = np.flatnonzero(segments[0] == segment)
        print(f"segment {segment}, columns {columns[0]}..{columns[-1]}: "
              f"{tokenizer.decode(tokens[0, columns])!r}")


packed token IDs: [1, 9690, 198, 6077, 5453, 30, 2, 198, 1, 4093, 198, 34, 1232, 216, 35, 47, 2, 198, 1, 520, 9531, 198, 37, 43, 45284, 2294, 30, 2, 198, 1, 9690, 198, 6077, 5453, 30, 2, 198, 1, 4093, 198, 34, 1232, 216, 35, 47, 2, 198, 1, 520, 9531, 198, 37, 43, 45284, 2294, 30, 2, 198, 1, 9690, 198, 6077, 5453, 30, 2, 198, 1, 4093, 198, 34, 1232, 216, 35, 47, 2, 198, 1, 520, 9531, 198, 37, 43, 45284, 2294, 30, 2, 198, 1, 9690, 198, 6077, 5453, 30, 2, 198, 1, 4093, 198, 34, 1232, 216, 35, 47, 2, 198, 1, 520, 9531, 198, 37, 43, 45284, 2294, 30, 2, 198, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
segment IDs:      [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

Repad those same conversations one per row to compare the two layouts. Both batches contain the same real tokens. Packing uses fewer padding slots, which means more of the model's work goes to conversation tokens.


In [8]:
columns_by_segment = [
    np.flatnonzero(segments[0] == segment)
    for segment in np.unique(segments[0]) if segment != 0
]
shape = (len(columns_by_segment), SEQUENCE_LENGTH + 1)
single_segments = np.zeros(shape, np.int32)
for row, columns in enumerate(columns_by_segment):
    single_segments[row, :len(columns)] = 1

for label, shown in (("packed", segments), ("one conversation a row", single_segments)):
    real = int((shown != 0).sum())
    total = shown.size
    print(f"{label:24} {real}/{total} tokens, fill {real / total:.1%}")


packed                   116/129 tokens, fill 89.9%
one conversation a row   116/516 tokens, fill 22.5%


## The loss and its gradient per token

Write the tokens as $x_1,\ldots,x_T$, their message roles as $r_t \in \{\mathrm{system}, \mathrm{user}, \mathrm{assistant}, \mathrm{tool}\}$, and their segment IDs as $s_t$. The model gives $\pi_\theta(x_t \mid x_{<t})$, with the context restricted to the same segment. Set $m_t = 1$ on scored assistant tokens and 0 elsewhere, including unused slots and assistant headers.

The loss is the mean over scored tokens in the batch. The sums include all rows:

$$
L(\theta) = -\frac{1}{\sum_t m_t}\sum_t m_t \log \pi_\theta(x_t \mid x_{<t}).
$$

If $z_t$ is the vector of logits predicting $x_t$, its gradient is

$$
\frac{\partial L}{\partial z_t}
= \bigl(\operatorname{softmax}(z_t) - \operatorname{onehot}(x_t)\bigr)
  \cdot \frac{m_t}{\sum_s m_s}.
$$

A masked logit has zero direct loss gradient. User tokens still affect the assistant predictions through attention, so they remain in the inputs.

`objective.predict` returns the objective's loss and the model's logits for this batch. We compute cross entropy from those logits in JAX, then compare the formula's gradient with `jax.grad`. No optimizer update happens here.


In [9]:
from dew.objectives.base import Step

step = Step(jnp.asarray(0), jax.random.key(SEED), None, training=False)
statistics, _, prediction = objective.predict(model.variables, batch, step, train=False)
logits = prediction.logits
targets, weights = jnp.asarray(tokens[:, 1:]), jnp.asarray(mask, jnp.float32)

def token_loss(z):
    return -jnp.take_along_axis(
        jax.nn.log_softmax(z), targets[..., None], axis=-1)[..., 0]

def mean_loss(z):
    return (token_loss(z) * weights).sum() / weights.sum()

hand_loss, dew_loss = mean_loss(logits), statistics.mean()[0]
hand_gradient = (jax.nn.softmax(logits) - jax.nn.one_hot(targets, logits.shape[-1]))
hand_gradient *= weights[..., None] / weights.sum()
autodiff_gradient = jax.grad(mean_loss)(logits)
np.testing.assert_array_equal(prediction.weights, mask)
np.testing.assert_allclose(hand_loss, dew_loss, rtol=1e-5, atol=1e-6)
np.testing.assert_allclose(hand_gradient, autodiff_gradient, rtol=1e-5, atol=1e-7)
print("scored token losses:", np.round(np.asarray(token_loss(logits))[mask], 3))
print(f"hand loss {float(hand_loss):.6f}, objective loss {float(dew_loss):.6f}")
print("maximum logit-gradient error:",
      float(jnp.max(jnp.abs(hand_gradient - autodiff_gradient))))


scored token losses: [ 5.081  9.086 17.733 14.18   1.335  0.214  0.     5.098  9.295 17.8
 14.079  1.399  0.192  0.     5.081  9.086 17.733 14.18   1.335  0.214
  0.     5.098  9.295 17.8   14.079  1.399  0.192  0.   ]
hand loss 6.820891, objective loss 6.820891
maximum logit-gradient error: 7.450580596923828e-09


## A short fine-tune

We write addition questions with replies such as "The sum is 7; Dew done." The fixed sign-off makes the style change easy to check. The system message asks for brevity; it never asks for the sign-off.

Three pairs are held out of the training records. We ask those questions before and after training with greedy decoding, so sampling noise cannot explain a change. This is a small style exercise, not an evaluation of arithmetic ability.


In [10]:
held_out_pairs = ((2, 5), (4, 3), (6, 2))[:HELD_OUT]
train_records = tuple(
    {"messages": [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"What is {a} plus {b}?"},
        {"role": "assistant", "content": f"The sum is {a + b}; {SIGN_OFF}"},
    ]}
    for a in range(1, MAX_NUMBER + 1)
    for b in range(1, MAX_NUMBER + 1)
    if (a, b) not in held_out_pairs
)
questions = [f"What is {a} plus {b}?" for a, b in held_out_pairs]
data = ChatData(
    train_records, tokenizer=CHECKPOINT, seq_len=SEQUENCE_LENGTH, long="drop",
).load(batch=BATCH_SIZE, read="memory", loading=Loading(workers=0), seed=SEED)
print(f"{len(train_records)} training conversations, {len(questions)} held-out questions")
print(f"{data.records} packed windows, {data.steps_per_epoch} batches per epoch")


Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

Fetching 11 files: 100%|██████████| 11/11 [00:00<00:00, 1575.51it/s]

61 training conversations, 3 held-out questions
21 packed windows, 5 batches per epoch


For generation, `add_generation_prompt=True` appends the next assistant header. There is no answer in the prompt. The generation task uses the checkpoint's end-of-turn token to stop and returns just the new text.


In [11]:
from dew.sampling import Sampling

prompts = [
    tokenizer.apply_chat_template([
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ], tokenize=False, add_generation_prompt=True)
    for question in questions
]
before_task = model.text_generation(sampling=Sampling(temperature=0))
before = list(before_task(prompts, MAX_NEW_TOKENS, key=SEED).text)
for question, answer in zip(questions, before, strict=True):
    print(question, "->", repr(answer))


02:35:30 WARNING dew.nn.kernels.generation: attention runs 'cudnn', not 'flash', the kernel measured fastest on sm80: the call has a mask


What is 2 plus 5? -> 'To find 2 plus 5, we simply add the two numbers together. 2 + 5 equals 7.'
What is 4 plus 3? -> '4 + 3 = 7.'
What is 6 plus 2? -> '6 + 2 is 8.'


The objective already holds the pretrained weights. `Trainer.fit` updates them with the assistant-only loss we checked above. It repeats the packed training data until `STEPS` updates have run. No moving average is configured, so the returned `state.variables` are the trained weights.


In [12]:
from dew import Trainer
from dew.config import OptimConfig

state = Trainer(
    objective, OptimConfig(learning_rate=LEARNING_RATE), key=SEED,
).fit(data, steps=STEPS, log_every=max(1, STEPS // 8))
print("updates:", int(state.updates))


Training CausalTransformer from step 0 to 80: 134.5M parameters, on 1 × NVIDIA A100-SXM4-40GB, batch 4, float32 parameters, bfloat16 compute

02:36:34 WARNING dew.nn.kernels.tuning: 6 kernel launches have no config measured on NVIDIA A100-SXM4-40GB, so they run the nearest measured shape's or their heuristic: measure them on a GPU to yourself with Trainer(tune_kernels=True) or `python tools/kernel_picks.py`


step 10/80  loss 0.007887  ce 0.007887  perplexity 1.008  token_accuracy 100.0%  data_wait_ms 0.02672  grad_norm 9.721  compiles 1.000  cache_hits 1.000  cache_misses 3.000  peak_memory_gib 2.281  step_time_ms 92.63  samples_per_sec 43.18  tokens 422.7  tokens_per_s 4,563  mfu 1.2%  hlo_flops_util 1.5%  hfu 1.5%  accepted 100.0%  learning_rate 1.000e-04  attempted/memory:42bae68eb20233f1dff06dbc0de63f3fef892526c9cc4628c7f8f15fbaa2038b 61.00  accepted/memory:42bae68eb20233f1dff06dbc0de63f3fef892526c9cc4628c7f8f15fbaa2038b 21.00  truncated/memory:42bae68eb20233f1dff06dbc0de63f3fef892526c9cc4628c7f8f15fbaa2038b 0  chunks_per_record/memory:42bae68eb20233f1dff06dbc0de63f3fef892526c9cc4628c7f8f15fbaa2038b 0  0:00:03 left

step 20/80  loss 0.008095  ce 0.008095  perplexity 1.008  token_accuracy 99.3%  data_wait_ms 0.02554  grad_norm 1.511  compiles 1.000  cache_hits 1.000  cache_misses 3.000  peak_memory_gib 2.281  step_time_ms 29.95  samples_per_sec 133.6  tokens 430.5  tokens_per_s 14,376  mfu 3.7%  hlo_flops_util 4.6%  hfu 4.6%  accepted 100.0%  learning_rate 1.000e-04  attempted/memory:42bae68eb20233f1dff06dbc0de63f3fef892526c9cc4628c7f8f15fbaa2038b 61.00  accepted/memory:42bae68eb20233f1dff06dbc0de63f3fef892526c9cc4628c7f8f15fbaa2038b 21.00  truncated/memory:42bae68eb20233f1dff06dbc0de63f3fef892526c9cc4628c7f8f15fbaa2038b 0  chunks_per_record/memory:42bae68eb20233f1dff06dbc0de63f3fef892526c9cc4628c7f8f15fbaa2038b 0  0:00:02 left

step 30/80  loss 0.001825  ce 0.001825  perplexity 1.002  token_accuracy 100.0%  data_wait_ms 0.02463  grad_norm 0.5489  compiles 1.000  cache_hits 1.000  cache_misses 3.000  peak_memory_gib 2.281  step_time_ms 32.35  samples_per_sec 123.6  tokens 415.4  tokens_per_s 12,840  mfu 3.3%  hlo_flops_util 4.2%  hfu 4.2%  accepted 100.0%  learning_rate 1.000e-04  attempted/memory:42bae68eb20233f1dff06dbc0de63f3fef892526c9cc4628c7f8f15fbaa2038b 61.00  accepted/memory:42bae68eb20233f1dff06dbc0de63f3fef892526c9cc4628c7f8f15fbaa2038b 21.00  truncated/memory:42bae68eb20233f1dff06dbc0de63f3fef892526c9cc4628c7f8f15fbaa2038b 0  chunks_per_record/memory:42bae68eb20233f1dff06dbc0de63f3fef892526c9cc4628c7f8f15fbaa2038b 0  0:00:02 left

...

step 70/80  loss 3.903e-05  ce 3.903e-05  perplexity 1.000  token_accuracy 100.0%  data_wait_ms 0.02360  grad_norm 0.003596  compiles 1.000  cache_hits 1.000  cache_misses 3.000  peak_memory_gib 2.281  step_time_ms 28.39  samples_per_sec 140.9  tokens 423.0  tokens_per_s 14,899  mfu 3.8%  hlo_flops_util 4.8%  hfu 4.8%  accepted 100.0%  learning_rate 1.000e-04  attempted/memory:42bae68eb20233f1dff06dbc0de63f3fef892526c9cc4628c7f8f15fbaa2038b 61.00  accepted/memory:42bae68eb20233f1dff06dbc0de63f3fef892526c9cc4628c7f8f15fbaa2038b 21.00  truncated/memory:42bae68eb20233f1dff06dbc0de63f3fef892526c9cc4628c7f8f15fbaa2038b 0  chunks_per_record/memory:42bae68eb20233f1dff06dbc0de63f3fef892526c9cc4628c7f8f15fbaa2038b 0  0:00:00 left

step 80/80  loss 2.341e-05  ce 2.341e-05  perplexity 1.000  token_accuracy 100.0%  data_wait_ms 0.02723  grad_norm 0.003128  compiles 1.000  cache_hits 1.000  cache_misses 3.000  peak_memory_gib 2.281  step_time_ms 33.17  samples_per_sec 120.6  tokens 422.9  tokens_per_s 12,748  mfu 3.3%  hlo_flops_util 4.1%  hfu 4.1%  accepted 100.0%  learning_rate 1.000e-04  attempted/memory:42bae68eb20233f1dff06dbc0de63f3fef892526c9cc4628c7f8f15fbaa2038b 61.00  accepted/memory:42bae68eb20233f1dff06dbc0de63f3fef892526c9cc4628c7f8f15fbaa2038b 21.00  truncated/memory:42bae68eb20233f1dff06dbc0de63f3fef892526c9cc4628c7f8f15fbaa2038b 0  chunks_per_record/memory:42bae68eb20233f1dff06dbc0de63f3fef892526c9cc4628c7f8f15fbaa2038b 0

Trained 80 steps in 0:01:27: first step after 84.77 s, then 31.9 step/s

2.8% of the wall time in steps, final loss 2.341e-05

updates: 80


Replace the bundle's weights with the trained ones and ask exactly the same questions. The table shows both replies. The last line counts replies ending in our sign-off; the CPU smoke run uses too few updates to learn it.


In [13]:
from dataclasses import replace
from itertools import zip_longest
from textwrap import wrap

after_task = replace(model, variables=state.variables).text_generation(
    sampling=Sampling(temperature=0))
after = list(after_task(prompts, MAX_NEW_TOKENS, key=SEED).text)
for question, old, new in zip(questions, before, after, strict=True):
    print("\n" + question)
    print(f"{'before':48} | after")
    for left, right in zip_longest(wrap(old, 48), wrap(new, 48), fillvalue=""):
        print(f"{left:48} | {right}")
old_count = sum(answer.strip().endswith(SIGN_OFF) for answer in before)
new_count = sum(answer.strip().endswith(SIGN_OFF) for answer in after)
print(f"fixed sign-off: before {old_count}/{len(questions)}, after {new_count}/{len(questions)}")



What is 2 plus 5?
before                                           | after
To find 2 plus 5, we simply add the two numbers  | The sum is 7; Dew done.
together. 2 + 5 equals 7.                        | 

What is 4 plus 3?
before                                           | after
4 + 3 = 7.                                       | The sum is 7; Dew done.

What is 6 plus 2?
before                                           | after
6 + 2 is 8.                                      | The sum is 8; Dew done.
fixed sign-off: before 0/3, after 3/3


## Where to go next

SFT changed the answer style by training on assistant completions. Packing kept separate conversations from sharing attention or next-token targets.

Notebook 10 moves to reinforcement learning with verifiable rewards (RLVR), where a check of the answer supplies the training signal. Notebook 11 uses a decoder in an agent that takes actions and reads tool results.
